# 01 · Bronze — dado cru, com rastreabilidade

```
landing (arquivos crus) ──► BRONZE (Iceberg) ──► Silver ──► Gold ─┬─► Postgres ──► Metabase
                                                                  └─► chunks ──► pgvector (próximo módulo)
```

**Regra da Bronze:** guardar como veio. Nada de limpar aqui. Só acrescentamos 3 colunas para saber *de onde* e *quando* veio:
`_ingest_ts`, `_source_file`, `_batch_id`.

**Antes de começar:** `docker compose up -d` (pasta `docker/`) e `pip install -r requirements.txt`.

## Passo 0 · O ambiente está de pé?

In [1]:
import sys, subprocess
r = subprocess.run([sys.executable, "../scripts/verifica_ambiente.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)

OK    Java 
OK    PySpark 3.5.3 
OK    MinIO (buckets landing e warehouse) 
OK    Spark 3.5.3 (grava e lÃª Parquet) 

Postgres + pgvector e Metabase: vamos montar juntos nas prÃ³ximas etapas.
Tudo pronto.
ÒXITO: o processo com PID 9496 (processo filho de PID 14708) foi finalizado.
ÒXITO: o processo com PID 14708 (processo filho de PID 9372) foi finalizado.
ÒXITO: o processo com PID 9372 (processo filho de PID 2540) foi finalizado.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 0:>                                                          (0 + 0) / 2]

[Stage 0:>                                                          (0 + 2) / 2]

                                                                                



## Passo 0.5 · Baixar as dependências do Spark (só na primeira vez)

O Spark precisa de alguns jars (Iceberg, S3A com o AWS SDK, JDBC). Na **primeira execução** ele os baixa da internet, e isso pode levar **vários minutos**, principalmente com a rede da sala dividida. Os arquivos ficam guardados em cache (pasta `.ivy2`) e, depois disso, o Spark abre rápido.

Rode a célula abaixo **em casa, antes da aula**. Ela abre um processo separado, baixa os jars e termina, sem deixar o Spark aberto neste notebook.

In [2]:
import sys, subprocess

r = subprocess.run([sys.executable, "../scripts/preparar_jars.py"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])      # fim da saída: deve terminar com "Jars baixados e prontos"
print(r.stderr[-1500:])      # se houver erro, a causa aparece aqui

:: loading settings :: url = jar:file:/C:/Users/User/Documents/Arque_coleta_dados/.venv/Lib/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Jars baixados e prontos. Spark 3.5.3
ÒXITO: o processo com PID 15352 (processo filho de PID 14280) foi finalizado.
ÒXITO: o processo com PID 14280 (processo filho de PID 14940) foi finalizado.
ÒXITO: o processo com PID 14940 (processo filho de PID 16348) foi finalizado.

pcEnv.onSuccess$1(NettyRpcEnv.scala:225)
	at org.apache.spark.rpc.netty.NettyRpcEnv.$anonfun$askAbortable$5(NettyRpcEnv.scala:239)
	at org.apache.spark.rpc.netty.NettyRpcEnv.$anonfun$askAbortable$5$adapted(NettyRpcEnv.scala:238)
	at scala.concurrent.impl.CallbackRunnable.run(Promise.scala:64)
	at org.apache.spark.util.ThreadUtils$$anon$1.execute(ThreadUtils.scala:99)
	at scala.concurrent.impl.ExecutionContextImpl$$anon$4.execute(ExecutionContextImpl.scala:138)
	at scala.concurrent.impl.CallbackRunnable.executeWithValue(Promise.scala:72)
	at sc

## Passo 1 · Abrir o Spark (modo local)

O Spark roda **dentro deste Python**, com `local[2]` (2 núcleos) e 2 GB. Na primeira vez ele baixa os jars (Iceberg, S3A, JDBC). O S3A traz junto o AWS SDK, que é grande, então pode levar vários minutos e precisa de internet. Os jars ficam em cache em ~/.ivy2 e nas próximas execuções é rápido.
Para mais folga, `SPARK_CORES` e `SPARK_MEMORY` (variáveis de ambiente antes de abrir o Jupyter).
Spark UI: <http://localhost:4040> enquanto houver um job rodando.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

Spark 3.5.3 | master: local[2] | storage: minio


## Passo 2 · Os arquivos chegam na landing (object storage)

In [4]:
n = L.upload_to_landing(L.DATA_DIR / "lote1", "mercadonordeste/lote1")
print(n, "arquivos enviados")
for chave, tam in L.list_landing("mercadonordeste/lote1"):
    print(f"{tam:>10,} bytes  {chave}")

4 arquivos enviados
   529,223 bytes  mercadonordeste/lote1/avaliacoes.csv
   420,891 bytes  mercadonordeste/lote1/clientes.csv
 7,601,232 bytes  mercadonordeste/lote1/pedidos.jsonl
     9,044 bytes  mercadonordeste/lote1/produtos.csv


Abra o console do MinIO (<http://localhost:9001>, `admin` / `admin12345`) e veja o bucket **landing**.
É um *object storage*: não há pastas de verdade, só chaves com `/` no nome. 
Você achou os objetos no bucket warehouse do MinIO, em lakehouse/bronze/<tabela>/data e metadata.

## Passo 3 · Landing → Bronze

In [5]:
contagens = M.ingest_lote(spark, "lote1", L.landing_uri)
contagens

{'clientes': 5000, 'produtos': 200, 'pedidos': 30150, 'avaliacoes': 6489}

In [6]:
spark.sql("SHOW TABLES IN lake.bronze").show()
spark.table("lake.bronze.pedidos").printSchema()
spark.table("lake.bronze.pedidos").select("pedido_id","cliente_id","status","itens","_batch_id","_source_file").show(3, truncate=60)

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|   bronze|  clientes|      false|
|   bronze|avaliacoes|      false|
|   bronze|  produtos|      false|
|   bronze|   pedidos|      false|
+---------+----------+-----------+

root
 |-- cliente_id: long (nullable = true)
 |-- data_pedido: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- itens: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- preco_unit: double (nullable = true)
 |    |    |-- produto_id: long (nullable = true)
 |    |    |-- qtd: long (nullable = true)
 |-- pedido_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)

+---------+----------+---------+--------------------------------------------------+---------+-----------

Repare: `itens` chegou como **lista de structs** (o JSON foi preservado) e os CSVs chegaram como texto. A tipagem é problema da Silver.

## Passo 4 · Idempotência
Rode o Passo 3 de novo. As contagens **não** dobram: a ingestão apaga o lote e regrava. Pipeline que duplica ao reexecutar é pipeline frágil.

In [7]:
antes = spark.table("lake.bronze.pedidos").count()
M.ingest_lote(spark, "lote1", L.landing_uri)
depois = spark.table("lake.bronze.pedidos").count()
print(antes, "->", depois, "| idempotente:", antes == depois)

30150 -> 30150 | idempotente: True


## Checkpoint
- [ ] `lake.bronze` tem `clientes`, `produtos`, `pedidos`, `avaliacoes`.
- [ ] Reexecutar não duplica.
- [ ] Você achou os objetos no bucket `warehouse` do MinIO (pastas `bronze/…/data` e `metadata`).

➡️ Próximo: `02_silver.ipynb`